# Notebook 02 — Chemical Space & Applicability Domain

**Input:** outputs dari notebook 01 (split train/test per endpoint + `combined_all_endpoints.csv`)  
**Requires:** no model, no GPU — runs entirely on CPU

| Section | Analysis | Output |
|---------|----------|--------|
| 1 | ECFP4 fingerprints | per-molecule 2048-dim bit vector |
| 2 | UMAP (fit train-only, transform test) | 6-panel scatter plot |
| 3 | Domain-shift metrics: CD, NND, CHC | heatmap |
| 4 | Tanimoto similarity distribution | histogram (primary quantitative criterion) |
| 5 | Scaffold diversity table | Table 2 |
| 6 | Physicochemical descriptor distributions | violin plots |

---
## 0 — Imports & Config

In [ ]:
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
from scipy.spatial import ConvexHull
from scipy.spatial.distance import cdist
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, MACCSkeys, Descriptors
from rdkit import RDLogger
import umap

warnings.filterwarnings('ignore')
RDLogger.DisableLog('rdApp.*')

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))
from utils.chem_utils import get_bemis_murcko_scaffold

with open(REPO_ROOT / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

SEED       = CFG['seed']
SPLIT_ROOT = Path(CFG['paths']['split_root'])
FIG_DIR    = Path(CFG['paths']['results_root']) / 'chemical_space'
FIG_DIR.mkdir(parents=True, exist_ok=True)

TASKS = [t['name'] for t in CFG['targets']]
TASK_LABELS = {t['name']: t['label'] for t in CFG['targets']}

# ECFP4 column names (2048-bit)
ECFP4_COLS = [f'ECFP4_{i}' for i in range(2048)]

print(f'Tasks : {TASKS}')
print(f'FIG_DIR: {FIG_DIR}')

---
## 1 — Compute ECFP4 Fingerprints

Fingerprints are computed fresh here from canonical SMILES in the split files.  
If you already ran notebook 03 (molecular representation), you can load the saved fingerprint CSVs instead.

In [ ]:
def smiles_to_ecfp4(smiles: str, radius: int = 2, nbits: int = 2048):
    """Returns 2048-dim ECFP4 bit vector or None."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    fp  = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nbits)
    arr = np.zeros(nbits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def load_split(task: str, split: str) -> pd.DataFrame:
    """Load train or test xlsx for a given task."""
    path = SPLIT_ROOT / task / f'{split}.xlsx'
    if not path.exists():
        raise FileNotFoundError(f'{path} not found — run notebook 01 first.')
    return pd.read_excel(path)


def get_fp_matrix(df: pd.DataFrame, smiles_col: str = 'canonical_smiles'):
    """Returns (X: np.ndarray, valid_idx: list) for ECFP4 from a DataFrame."""
    results = df[smiles_col].apply(smiles_to_ecfp4)
    valid   = results.dropna()
    X       = np.stack(valid.values).astype(np.float32)
    return X, valid.index.tolist()


# Pre-load all splits + compute fingerprints
fp_data = {}  # task -> {train: {X, df}, test: {X, df}}

for task in TASKS:
    fp_data[task] = {}
    for split in ['train', 'test']:
        df       = load_split(task, split)
        # Normalise SMILES column name (nb01 may save as canonical_smiles or Smiles)
        smi_col  = 'canonical_smiles' if 'canonical_smiles' in df.columns else 'Smiles'
        X, idx   = get_fp_matrix(df, smi_col)
        fp_data[task][split] = {'X': X, 'df': df.loc[idx].reset_index(drop=True)}
    n_tr = len(fp_data[task]['train']['df'])
    n_te = len(fp_data[task]['test']['df'])
    print(f'  [{task}] train={n_tr} | test={n_te}')

print('\n✅ Fingerprints ready.')

---
## 2 — UMAP Embedding

UMAP is **fit on train only**, then `.transform()` applied to test.  
This prevents test-set information from influencing the projection.

In [ ]:
umap_results = {}  # task -> {emb_train, emb_test, y_train, y_test}

for task in TASKS:
    X_tr = fp_data[task]['train']['X']
    X_te = fp_data[task]['test']['X']
    y_tr = fp_data[task]['train']['df']['label'].values
    y_te = fp_data[task]['test']['df']['label'].values

    reducer = umap.UMAP(
        n_components=2, n_neighbors=15, min_dist=0.1,
        metric='euclidean', random_state=SEED
    )
    emb_tr = reducer.fit_transform(X_tr)
    emb_te = reducer.transform(X_te)     # ← fit on train, transform test only

    umap_results[task] = dict(
        emb_train=emb_tr, emb_test=emb_te,
        y_train=y_tr, y_test=y_te
    )
    print(f'  [{task}] UMAP done')

print('\n✅ UMAP embeddings computed.')

In [ ]:
# ── Figure 1: 6-panel UMAP (train/test split origin) ─────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

SPLIT_PALETTE = {'Train': '#3B8BD4', 'Test': '#E8593C'}

for i, task in enumerate(TASKS):
    ax   = axes[i]
    res  = umap_results[task]

    ax.scatter(res['emb_train'][:, 0], res['emb_train'][:, 1],
               c=SPLIT_PALETTE['Train'], s=6, alpha=0.4, label='Train', rasterized=True)
    ax.scatter(res['emb_test'][:, 0],  res['emb_test'][:, 1],
               c=SPLIT_PALETTE['Test'],  s=6, alpha=0.7, label='Test',  rasterized=True)

    ax.set_title(TASK_LABELS[task], fontsize=11, fontweight='bold')
    ax.set_xlabel('UMAP 1', fontsize=9)
    ax.set_ylabel('UMAP 2', fontsize=9)
    ax.tick_params(labelsize=8)
    if i == 0:
        ax.legend(fontsize=9, markerscale=2)

fig.suptitle('Chemical Space — Train vs Held-out Test (ECFP4 UMAP)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'umap_train_test.png', dpi=200, bbox_inches='tight')
plt.savefig(FIG_DIR / 'umap_train_test.pdf', bbox_inches='tight')
plt.show()

# ── Figure 1b: 6-panel UMAP (active/non-active, train only) ──────────────────
LABEL_PALETTE = {1: '#E8593C', 0: '#3B8BD4'}
LABEL_NAMES   = {1: 'Active', 0: 'Non-active'}

fig2, axes2 = plt.subplots(2, 3, figsize=(15, 9))
axes2 = axes2.flatten()

for i, task in enumerate(TASKS):
    ax  = axes2[i]
    res = umap_results[task]
    for lbl in [0, 1]:
        mask = res['y_train'] == lbl
        ax.scatter(res['emb_train'][mask, 0], res['emb_train'][mask, 1],
                   c=LABEL_PALETTE[lbl], s=6, alpha=0.5,
                   label=LABEL_NAMES[lbl], rasterized=True)
    ax.set_title(TASK_LABELS[task], fontsize=11, fontweight='bold')
    ax.set_xlabel('UMAP 1', fontsize=9)
    ax.set_ylabel('UMAP 2', fontsize=9)
    ax.tick_params(labelsize=8)
    if i == 0:
        ax.legend(fontsize=9, markerscale=2)

fig2.suptitle('Chemical Space — Active vs Non-active (Train Pool)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'umap_active_inactive.png', dpi=200, bbox_inches='tight')
plt.savefig(FIG_DIR / 'umap_active_inactive.pdf', bbox_inches='tight')
plt.show()

---
## 3 — Domain-Shift Metrics: CD, NND, CHC

Computed in the UMAP-embedded space. Interpreted as qualitative indicators —  
Tanimoto similarity (section 4) is the primary quantitative criterion.

- **CD** (Centroid Distance): Euclidean distance between train/test centroids  
- **NND** (Nearest-Neighbor Distance): mean distance from each test point to its nearest train neighbour  
- **CHC** (Convex-Hull Coverage): fraction of test points inside the convex hull of train embeddings

In [ ]:
def centroid_distance(tr, te):
    return float(np.linalg.norm(tr.mean(0) - te.mean(0)))

def nearest_neighbor_distance(tr, te):
    return float(cdist(te, tr).min(axis=1).mean())

def convex_hull_coverage(tr, te):
    try:
        hull = ConvexHull(tr)
        A, b = hull.equations[:, :-1], hull.equations[:, -1]
        inside = np.all((A @ te.T).T + b <= 1e-7, axis=1)
        return float(inside.mean())
    except Exception:
        return np.nan


metrics_rows = []
for task in TASKS:
    tr = umap_results[task]['emb_train']
    te = umap_results[task]['emb_test']
    row = {
        'Endpoint': TASK_LABELS[task],
        'n_train':  len(tr),
        'n_test':   len(te),
        'CD':  centroid_distance(tr, te),
        'NND': nearest_neighbor_distance(tr, te),
        'CHC': convex_hull_coverage(tr, te),
    }
    metrics_rows.append(row)
    print(f'  [{task}] CD={row["CD"]:.4f} | NND={row["NND"]:.4f} | CHC={row["CHC"]:.4f}')

metrics_df = pd.DataFrame(metrics_rows).set_index('Endpoint')
print('\n', metrics_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    metrics_df[['CD', 'NND', 'CHC']].astype(float),
    annot=True, fmt='.3f', cmap='viridis', ax=ax,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'metric value'}
)
ax.set_title('Domain-Shift Metrics (UMAP space)\nTrain pool vs Held-out test',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'domain_shift_heatmap.png', dpi=200, bbox_inches='tight')
plt.savefig(FIG_DIR / 'domain_shift_heatmap.pdf', bbox_inches='tight')
plt.show()

---
## 4 — Tanimoto Similarity Distribution

Primary quantitative criterion for chemical space separation.  
For each test molecule, compute max Tanimoto similarity to any training molecule (ECFP4).  
Low similarity = structurally novel test set relative to training — consistent with scaffold split.

In [ ]:
from rdkit.DataStructs import BulkTanimotoSimilarity

def get_rdkit_fps(df: pd.DataFrame, smiles_col: str = 'canonical_smiles'):
    """Returns list of RDKit fingerprint objects (for Tanimoto)."""
    fps = []
    for smi in df[smiles_col]:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is not None:
            fps.append(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048))
        else:
            fps.append(None)
    return fps


fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
tanimoto_rows = []

for i, task in enumerate(TASKS):
    smi_col  = 'canonical_smiles' if 'canonical_smiles' in fp_data[task]['train']['df'].columns else 'Smiles'
    fps_tr   = [fp for fp in get_rdkit_fps(fp_data[task]['train']['df'], smi_col) if fp is not None]
    fps_te   = [fp for fp in get_rdkit_fps(fp_data[task]['test']['df'],  smi_col) if fp is not None]

    # Max Tanimoto similarity to any train molecule
    max_sim  = np.array([max(BulkTanimotoSimilarity(fp, fps_tr)) for fp in fps_te])

    tanimoto_rows.append({
        'Endpoint': TASK_LABELS[task],
        'mean_max_tanimoto': float(max_sim.mean()),
        'median_max_tanimoto': float(np.median(max_sim)),
        'frac_below_0.4': float((max_sim < 0.4).mean()),
    })

    ax = axes[i]
    ax.hist(max_sim, bins=40, color='#3B8BD4', edgecolor='white', linewidth=0.4)
    ax.axvline(0.4, color='#E8593C', linestyle='--', linewidth=1.2, label='Sim = 0.4')
    ax.set_title(TASK_LABELS[task], fontsize=10, fontweight='bold')
    ax.set_xlabel('Max Tanimoto (test → train)', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.tick_params(labelsize=8)
    if i == 0:
        ax.legend(fontsize=8)

fig.suptitle('Tanimoto Similarity: Test Compounds vs Training Set (ECFP4)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'tanimoto_similarity.png', dpi=200, bbox_inches='tight')
plt.savefig(FIG_DIR / 'tanimoto_similarity.pdf', bbox_inches='tight')
plt.show()

tan_df = pd.DataFrame(tanimoto_rows).set_index('Endpoint')
print(tan_df.round(4).to_string())

---
## 5 — Scaffold Diversity (Table 2)

Unique Bemis-Murcko scaffolds per split, scaffold:molecule ratio, and overlap (should be 0%).

In [ ]:
scaffold_rows = []

for task in TASKS:
    for split in ['train', 'test']:
        df      = fp_data[task][split]['df']
        smi_col = 'canonical_smiles' if 'canonical_smiles' in df.columns else 'Smiles'

        # If scaffold column already exists (from nb01), use it; otherwise recompute
        if 'scaffold' in df.columns:
            scaffolds = df['scaffold'].dropna()
        else:
            scaffolds = df[smi_col].apply(get_bemis_murcko_scaffold).dropna()

        fp_data[task][split]['scaffolds'] = set(scaffolds)

    scaf_tr    = fp_data[task]['train']['scaffolds']
    scaf_te    = fp_data[task]['test']['scaffolds']
    n_overlap  = len(scaf_tr & scaf_te)           # must be 0
    n_tr_mol   = len(fp_data[task]['train']['df'])
    n_te_mol   = len(fp_data[task]['test']['df'])

    scaffold_rows.append({
        'Endpoint':           TASK_LABELS[task],
        'Train mols':         n_tr_mol,
        'Train scaffolds':    len(scaf_tr),
        'Scaf/mol (train)':   round(len(scaf_tr) / max(n_tr_mol, 1), 3),
        'Test mols':          n_te_mol,
        'Test scaffolds':     len(scaf_te),
        'Scaf/mol (test)':    round(len(scaf_te) / max(n_te_mol, 1), 3),
        'Scaffold overlap':   n_overlap,
        'Overlap %':          round(100 * n_overlap / max(len(scaf_te), 1), 2),
    })

scaf_df = pd.DataFrame(scaffold_rows).set_index('Endpoint')

print('=' * 70)
print('TABLE 2 — Scaffold Diversity')
print('=' * 70)
print(scaf_df.to_string())

# Confirm leakage is still 0
assert (scaf_df['Scaffold overlap'] == 0).all(), 'Scaffold overlap detected!'
print('\n✅ Zero scaffold overlap confirmed.')

---
## 6 — Physicochemical Descriptor Distributions

MW, logP, TPSA, HBD, HBA per endpoint — violin plots split by class label (train only).

In [ ]:
PHYS_PROPS = {
    'MW':   Descriptors.MolWt,
    'logP': Descriptors.MolLogP,
    'TPSA': Descriptors.TPSA,
    'HBD':  Descriptors.NumHDonors,
    'HBA':  Descriptors.NumHAcceptors,
}

def compute_phys(df: pd.DataFrame, smi_col: str = 'canonical_smiles') -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        mol = Chem.MolFromSmiles(str(row[smi_col]))
        if mol is None:
            continue
        r = {'label': row['label'], 'label_name': row.get('label_name', str(row['label']))}
        for name, fn in PHYS_PROPS.items():
            r[name] = fn(mol)
        rows.append(r)
    return pd.DataFrame(rows)


# Compute and plot per endpoint (train only)
fig, axes = plt.subplots(len(PHYS_PROPS), len(TASKS),
                          figsize=(4 * len(TASKS), 3 * len(PHYS_PROPS)),
                          sharey='row')

PALETTE = {'Active': '#E8593C', 'Non-active': '#3B8BD4'}

for j, task in enumerate(TASKS):
    df      = fp_data[task]['train']['df']
    smi_col = 'canonical_smiles' if 'canonical_smiles' in df.columns else 'Smiles'
    phys_df = compute_phys(df, smi_col)

    # Map label int to name if needed
    if 'label_name' not in phys_df.columns or phys_df['label_name'].isnull().all():
        phys_df['label_name'] = phys_df['label'].map({1: 'Active', 0: 'Non-active'})

    for i, prop in enumerate(PHYS_PROPS):
        ax = axes[i][j]
        sns.violinplot(
            data=phys_df, x='label_name', y=prop, ax=ax,
            palette=PALETTE, order=['Active', 'Non-active'],
            inner='box', linewidth=0.8
        )
        ax.set_xlabel('')
        ax.set_ylabel(prop if j == 0 else '', fontsize=9)
        ax.tick_params(labelsize=7)
        if i == 0:
            ax.set_title(TASK_LABELS[task], fontsize=9, fontweight='bold')

fig.suptitle('Physicochemical Descriptor Distributions — Active vs Non-active (Train Pool)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'physchem_distributions.png', dpi=200, bbox_inches='tight')
plt.savefig(FIG_DIR / 'physchem_distributions.pdf', bbox_inches='tight')
plt.show()

---
## 7 — Save Summary Tables

In [ ]:
metrics_df.to_csv(FIG_DIR / 'domain_shift_metrics.csv')
scaf_df.to_csv(FIG_DIR    / 'scaffold_diversity.csv')
tan_df.to_csv(FIG_DIR     / 'tanimoto_similarity.csv')

print('Saved:')
for f in sorted(FIG_DIR.glob('*')):
    print(f'  {f.name}')

---
## ✅ Notebook 02 Complete

**Outputs** (all in `results/chemical_space/`):
```
umap_train_test.png/pdf
umap_active_inactive.png/pdf
domain_shift_heatmap.png/pdf    → Figure 2a
tanimoto_similarity.png/pdf     → Figure 2b  
physchem_distributions.png/pdf
domain_shift_metrics.csv
scaffold_diversity.csv          → Table 2
tanimoto_similarity.csv
```

➡️ **Next:** [03_molecular_representation.ipynb](03_molecular_representation.ipynb)